# Comparison Across Training Seeds

This notebook compares final-evaluation results across training seeds `156`, `157`, and `158` for each task, using experiments `001–003`.

The related unseen split (`test_unseen_related`) is excluded for object and object-region classification because these results are not scored. `validation_unseen_matched` is included for reference and is not an independent test result.

It exports one CSV summary table per task, containing individual seed values and summary statistics.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [ ]:
RESULTS_DIR = Path("../results/final_evaluation")
OUTPUT_DIR = Path("../exports/seed_comparison")
EXPERIMENT_NUMBERS = [1, 2, 3]
SEEDS = [156, 157, 158]

EVALUATION_SPLITS = [
    "test",
    "test_unseen_matched",
    "test_unseen_related",
    "validation_unseen_matched",
]

TASK_METRICS = {
    "object_classification": ["accuracy", "macro_f1_avg"],
    "object_region_classification": ["accuracy", "macro_f1_avg"],
    "force_level_classification": ["accuracy", "macro_f1_avg"],
    "object_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "object_region_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "force_n_regression": ["mae", "rmse", "r2"],
    "fsr_voltage_regression": ["mae", "rmse", "r2"],
}

UNSCORED_RELATED_TASKS = {
    "object_classification",
    "object_region_classification",
    "object_conditioned_classification",
    "object_region_conditioned_classification",
}

In [ ]:
def load_seed_results(task):
    """
    Load one final-evaluation run per experiment and training seed.

    Args:
        task (str): The task folder under the final-evaluation results directory.

    Returns:
        pd.DataFrame: The selected result rows indexed in training-seed order.
    """

    results_path = RESULTS_DIR / task / "experiments.parquet"
    results = pd.read_parquet(results_path)
    results = results.loc[results["experiment_number"].isin(EXPERIMENT_NUMBERS)].copy()

    # Keep seed values and summary columns in the same order across all task tables
    return results.set_index("seed").loc[SEEDS]


def calculate_seed_summary(task, results):
    """
    Calculate per-split metric statistics across the three training seeds.

    Args:
        task (str): The task folder identifying the metrics and scored splits.
        results (pd.DataFrame): The selected result rows indexed by training seed.

    Returns:
        pd.DataFrame: Individual seed values and summary statistics per metric.
    """

    rows = []
    for split in EVALUATION_SPLITS:
        # Skip unscored classification results while retaining force-task metrics
        if split == "test_unseen_related" and task in UNSCORED_RELATED_TASKS:
            continue

        for metric in TASK_METRICS[task]:
            column = f"{split}_{metric}"
            values = results.loc[SEEDS, column]
            row = {"split": split, "metric": metric}
            row.update({f"seed_{seed}": values.loc[seed] for seed in SEEDS})
            row.update(
                {
                    "n": values.count(),
                    "mean": values.mean(),
                    "std": values.std(ddof=1),
                    "min": values.min(),
                    "max": values.max(),
                }
            )
            rows.append(row)

    return pd.DataFrame(rows)


def export_seed_comparisons():
    """
    Export a seed-comparison CSV table for each final-evaluation task.

    Returns:
        dict[str, pd.DataFrame]: The summary table for each task.
    """

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    comparisons = {}
    for task in TASK_METRICS:
        results = load_seed_results(task)
        comparison = calculate_seed_summary(task, results)
        comparison.to_csv(OUTPUT_DIR / f"{task}.csv", index=False)
        comparisons[task] = comparison

    return comparisons

In [ ]:
# Run all task comparisons and retain the tables for display below
seed_comparisons = export_seed_comparisons()

In [ ]:
# Display each task table
with pd.option_context("display.max_rows", None, "display.precision", 4):
    for task, comparison in seed_comparisons.items():
        display(task)
        display(comparison)